<a href="https://colab.research.google.com/github/leweieeeee/Transformer-Warm-Start-ADAPT-VQE/blob/main/H2_ADAPT_VQE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%matplotlib inline

In [ ]:
%pip install pennylane
%pip install pandas

In [ ]:
%pip install pipreqs

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 43.5 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=84779735437bfcab0820713aa82e913ee5018738dd08cacc2307b74c5ce94542
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
Successfully built docopt
  Attempting uninstall: ipython
    Found existing installation: ipython 7.34.0
    Uninstalling ipython-7.34.0:
      Successfully uninstalled ipython-7.34.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipython==7.34.0, but you have ipython 8.12.3 which is incompatible.


In [ ]:

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import pennylane as qml

from pennylane import RandomLayers
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import numpy as np
import time
import pandas as pd

from pennylane import qchem

In [ ]:
x


Python 3.12.11


In [ ]:
!pipreqs . --force

INFO: Not scanning for jupyter notebooks.
INFO: Successfully saved requirements file in ./requirements.txt


In [ ]:
!pip freeze > requirements.txt

In [ ]:
def generate_data(bond_lengths, active_electrons=2, active_orbitals=2, basis="sto-3g"):
    data = []

    for L in bond_lengths:
        symbols = ["H", "H"]
        geometry = np.array([[0.0, 0.0, 0.0], [0.0, 0.0, L]],dtype=float)
        molecule = qchem.Molecule(symbols, geometry)

        H, qubits = qchem.molecular_hamiltonian(
            molecule,
            active_electrons=active_electrons,
            active_orbitals=active_orbitals,
        )

        singles, doubles = qchem.excitations(active_electrons, qubits)

        data.append({
            "bond_length": float(L),
            "molecule": molecule,
            "hamiltonian": H,
            "qubits": qubits,
            "singles": singles,
            "doubles": doubles,
        })

    return data


In [ ]:
def adapt_vqe(H, qubits, singles, doubles, max_iter = 20):
    active_electrons=2
    hf_state = qchem.hf_state(active_electrons, qubits)
    dev = qml.device("default.qubit", wires=qubits)
    @qml.qnode(dev)
    def circuit():
        [qml.PauliX(i) for i in np.nonzero(hf_state)[0]]
        return qml.expval(H)

    singles_excitations = [qml.SingleExcitation(0.0, x) for x in singles]
    doubles_excitations = [qml.DoubleExcitation(0.0, x) for x in doubles]
    operator_pool = doubles_excitations + singles_excitations
    opt = qml.optimize.AdaptiveOptimizer()
    records = []

    for step in range(len(operator_pool)):
        start_time = time.time()

        circuit, energy, gradient = opt.step_and_cost(circuit, operator_pool)

        elapsed_time = time.time() - start_time

        print("n = {:},  E = {:.8f} H, Largest Gradient = {:.3f}".format(step, energy, gradient))
        print(qml.draw(circuit, decimals=None)())
        print()

        tape = circuit.construct([], {})

        for op in tape.operations:
          print(op.name, op.wires, op.parameters)

        for idx, op in enumerate (tape.operations):
          record = {
            "step":step,
            "gate_type": op.name,
            "depth":idx,
            "energy": energy,
            "time": elapsed_time
            }
          if len(op.parameters) > 0:
            record["param"] = float(np.asarray(op.parameters[0]))
          else :
            record["param"] = None
          records.append(record)

        if gradient < 1e-4:
          break
    df = pd.DataFrame(records)
    return df


In [ ]:
bond_lengths = jnp.linspace(0.3, 4, 10)
dataset = generate_data(bond_lengths)
dfs = []

for entry in dataset:
    L        = entry["bond_length"]
    H        = entry["hamiltonian"]
    qubits   = entry["qubits"]
    singles  = entry["singles"]
    doubles  = entry["doubles"]
    df = adapt_vqe(H, qubits, singles, doubles)
    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)

combined_df.to_csv("combined_adapt_vqe_dataset.csv",index = False)


n = 0,  E = 0.80747505 H, Largest Gradient = 0.157
0: ──X─╭G²─┤ ╭<𝓗>
1: ──X─├G²─┤ ├<𝓗>
2: ────├G²─┤ ├<𝓗>
3: ────╰G²─┤ ╰<𝓗>

PauliX Wires([np.int64(0)]) []
PauliX Wires([np.int64(1)]) []
DoubleExcitation Wires([0, 1, 2, 3]) [tensor(0.08034896, requires_grad=True)]
n = 1,  E = 0.80116052 H, Largest Gradient = 0.000
0: ──X─╭G²─╭G²─┤ ╭<𝓗>
1: ──X─├G²─├G²─┤ ├<𝓗>
2: ────├G²─├G²─┤ ├<𝓗>
3: ────╰G²─╰G²─┤ ╰<𝓗>

PauliX Wires([np.int64(0)]) []
PauliX Wires([np.int64(1)]) []
DoubleExcitation Wires([0, 1, 2, 3]) [tensor(0.08034896, requires_grad=True)]
DoubleExcitation Wires([0, 1, 2, 3]) [tensor(-1.74520452e-17, requires_grad=True)]
n = 0,  E = -0.85179013 H, Largest Gradient = 0.164
0: ──X─╭G²─┤ ╭<𝓗>
1: ──X─├G²─┤ ├<𝓗>
2: ────├G²─┤ ├<𝓗>
3: ────╰G²─┤ ╰<𝓗>

PauliX Wires([np.int64(0)]) []
PauliX Wires([np.int64(1)]) []
DoubleExcitation Wires([0, 1, 2, 3]) [tensor(0.11370558, requires_grad=True)]
n = 1,  E = -0.86109978 H, Largest Gradient = 0.000
0: ──X─╭G²─╭G²─┤ ╭<𝓗>
1: ──X─├G²─├G²─┤ ├<𝓗>
2: ────├G²─├

In [ ]:
df_store = df.drop(columns = ["energy","time"])
df_store.to_csv("store_adapt_vqe_dataset.csv", index=False)